# Cloud recognition from frozen SEVIRI encoders

Questo esperimento verifica quanto le rappresentazioni apprese dai diversi obiettivi self-supervised permettano di riconoscere la copertura nuvolosa. Il target è il prodotto binario **`cma`** generato dal nuovo codice `CMA_original`: ciascun pixel valido è classificato come **sereno (0)** oppure **nuvoloso (1)**. Il valore 255 identifica un target mancante, escluso dalla loss e dalle metriche.

Il probe restituisce una probabilità di nube per ciascuno dei 224 × 224 pixel dell’ultimo frame osservato. L’encoder riceve i primi **10 frame**, distanziati di cinque minuti, della clip canonica di 16 frame; il target corrisponde al **frame 9**, cioè 45 minuti dopo l’inizio della clip. Il suo campo di input comprende tutti i sette canali SEVIRI, con la stessa normalizzazione globale usata nel rispettivo pretraining.

I punteggi misurano l’accordo con questa implementazione CMA su osservazioni di giornate distinte dal pretraining. Il confronto con un encoder casuale della stessa architettura quantifica il contributo del pretraining alla segmentazione. La verifica della corrispondenza con le nuvole atmosferiche reali richiederà anche osservazioni di riferimento indipendenti.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/emma_gpm').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Aprire il notebook da una cartella del repository deep_convection.')
sys.path.insert(0, str(ROOT / 'src'))
from emma_gpm.cloud_probe_reporting import load_results, plot_comparison, plot_paired_gain, plot_budget_gain, plot_examples

active_path = ROOT / 'results/cloud_probe_active_series.json'
active = json.loads(active_path.read_text()) if active_path.exists() else {}
default_results = Path(active.get('output_dir', 'results/cloud_mask_probe_20261031'))
if not default_results.is_absolute():
    default_results = ROOT / default_results
RESULTS = Path(os.environ.get('SEVIRI_CLOUD_PROBE_RESULTS', default_results))
queue_path = RESULTS / 'queue_identity.json'
queue = json.loads(queue_path.read_text()) if queue_path.exists() else {}
BASELINE_RESULTS = Path(queue['extend_from_series']) if queue.get('extend_from_series') else None
print('Epoche previste per la testa:', queue.get('plan', {}).get('epochs', '—'))
DATASET = Path(os.environ.get('SEVIRI_CLOUD_PROBE_DATASET', '/data1/deep_convection_data/cloud_mask_probe_20261031'))
pd.options.display.float_format = '{:.3f}'.format
results = load_results(RESULTS)
print(f'Probe completati: {len(results)}')
if (RESULTS / 'progress.json').exists():
    status = json.loads((RESULTS / 'progress.json').read_text())
    print('Stato della serie:', status['status'])
    print('Modello in elaborazione:', status.get('current') or '—')
    print('Encoder ancora attesi:', len(status.get('pending', [])))

## Osservazioni e costruzione delle maschere

Il dataset iniziale comprende **448 clip di training**, tratte dal 13 e 14 dicembre 2020, **112 clip di validation**, del 16 dicembre, e **112 clip di test**, del 26 dicembre. Il programma controlla che queste giornate siano separate dalle date dei manifest di pretraining e validation di ogni encoder. Tra i gruppi del probe è presente almeno un’intera giornata inutilizzata.

Ogni istante selezionato fornisce 14 ritagli spaziali disgiunti. Gli inizi delle clip sono distanziati di 80 minuti nel training e di 160 minuti nella validation e nel test. Le clip della stessa giornata rappresentano strutture atmosferiche correlate; la valutazione iniziale dispone di una giornata di validation e una di test.

CMA elabora l’intero dominio disponibile di **763 × 1762 pixel**, procedendo cronologicamente ogni cinque minuti. La prima maschera selezionata segue un’ora di osservazioni; lo stato temporale dell’algoritmo viene inizializzato all’inizio di ciascuna giornata. La maschera del frame 9 viene conservata prima di elaborare i frame successivi. Il ritaglio avviene dopo la classificazione sul dominio completo.

La geometria satellitare deriva dalle coordinate dell’archivio. La distinzione terra–mare usa i poligoni Natural Earth alla scala 1:10 milioni, valutati nel centro del pixel. I riferimenti radiometrici superficiali e atmosferici sono stimati dall’implementazione Pure-IR a sette canali. Questa prima configurazione usa i valori predefiniti di quota pari a zero e assenza di un campo ausiliario di neve e ghiaccio: terreno montuoso e neve invernale sono quindi aspetti da considerare nell’interpretazione delle maschere.

Nel grafico seguente la frazione nuvolosa è la percentuale di pixel validi classificati come nube dal CMA, aggregata separatamente per ciascuno split.

In [ ]:
if (DATASET / 'manifest.json').exists():
    manifest = json.loads((DATASET / 'manifest.json').read_text())
    samples = manifest['samples']
    fractions = []
    for split in ['train', 'validation', 'test']:
        subset = [r for r in samples if r['split'] == split]
        fractions.append(100 * sum(r['cloud_pixels'] for r in subset) / sum(r['valid_pixels'] for r in subset))
    fig, ax = plt.subplots(figsize=(6, 3.5), layout='constrained')
    bars = ax.bar(['Training', 'Validation', 'Test'], fractions, color=['#367fb0', '#d28e3b', '#4a9a73'])
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Cloud-labelled pixels (%)')
    ax.set_title('CMA cloud fraction')
    plt.show()
else:
    print('Dataset in preparazione; il grafico apparirà al completamento.')

## Testa di segmentazione e selezione del modello

Tutti i pesi dell’encoder sono congelati. Le rappresentazioni vengono estratte una sola volta, in modalità evaluation, e salvate su disco. Il programma verifica mediante checksum che pesi e buffer dell’encoder restino identici durante l’estrazione. Il calcolo usa BF16 sulla CPU e conserva le rappresentazioni in array float32; i training self-supervised continuano sulla GPU.

Si usano i token dell’ultimo intervallo temporale dell’encoder, disposti su una griglia di **14 × 14 posizioni**. Attraverso l’attenzione dell’encoder, ciascun token contiene informazione proveniente dalla storia osservata. La testa applica LayerNorm, un layer Linear con 128 unità, GELU e un secondo Linear con 256 uscite. Le 256 uscite descrivono i 16 × 16 pixel della patch; assemblando tutte le patch si ottiene la mappa di probabilità a 224 × 224 pixel.

La prima serie ha addestrato soltanto questa testa per **30 epoche**. L’estensione corrente porta il budget a **150 epoche complessive**, con batch size 16, AdamW, learning rate 0,001 e weight decay 0,0001. Le teste già completate riprendono dal checkpoint dell’epoca 30, conservando lo stato dell’optimizer e la storia delle metriche. Le rappresentazioni congelate vengono riutilizzate. Le teste per encoder resi disponibili successivamente ricevono tutte le 150 epoche a partire dall’inizializzazione. La loss è la binary cross-entropy sui pixel con target valido. I diversi encoder condividono gli split, il seed della testa e l’ordine dei minibatch. L’inizializzazione della testa è identica tra encoder della stessa architettura; Small e Base hanno dimensioni di ingresso differenti.

Per ciascuna combinazione di architettura, normalizzazione e seed iniziale viene addestrata anche la testa sopra un **encoder casuale**, indicato con **Random**. Il suo confronto appaiato usa la stessa procedura del modello preaddestrato. La validation seleziona l’epoca con la maggiore media tra IoU della nube e IoU del sereno; in caso di parità si conserva la prima epoca. La soglia di classificazione della probabilità di nube è fissata a **0,5**. Il test viene valutato dopo questa selezione.

## Confronto dei modelli

**Cloud IoU** confronta l’area di sovrapposizione tra nube predetta e nube target con l’area complessivamente occupata dalle due maschere. **Cloud Dice** misura anch’esso la sovrapposizione spaziale, attribuendo peso doppio ai pixel condivisi nel confronto con la somma delle due aree. I valori mostrati derivano dai conteggi dei pixel validi aggregati sull’intero test; un valore più alto indica un accordo maggiore con CMA.

Le legende indicano l’obiettivo del pretraining: **MVM** ricostruisce parti mascherate della storia; **Future** predice i frame futuri; **Joint** combina ricostruzione mascherata e previsione; **Residual** predice la correzione radiometrica rispetto alla previsione ottenuta per advezione. **Random** identifica il controllo con encoder inizializzato casualmente. Il gruppo della barra Random indica il modello cui il controllo è appaiato; i pesi del suo encoder restano quelli dell’inizializzazione. Ogni gruppo sull’asse orizzontale specifica architettura, numero di clip di pretraining e numero di epoche. Small con 16.384 clip ha ricevuto 75 epoche; gli altri esperimenti elencati ne hanno ricevute 150.

In [ ]:
results = load_results(RESULTS)
plot_comparison(results)
plt.show()

Il grafico successivo mostra, per ciascun modello preaddestrato, la differenza di Cloud IoU rispetto al proprio controllo casuale. L’unità è il **punto percentuale**: valori positivi indicano un vantaggio del pretraining con la stessa procedura di addestramento della testa. I confronti appaiono quando entrambi i probe sono terminati.

In [ ]:
plot_paired_gain(results)
plt.show()

## Effetto delle epoche aggiuntive

I grafici confrontano lo stesso encoder dopo il budget iniziale di 30 epoche e quello esteso a 150. Per ogni budget viene usato il checkpoint selezionato tramite mean IoU di validation. Nel pannello superiore, una differenza positiva di Cloud IoU sul test indica un miglioramento con il budget maggiore. Nel pannello inferiore, una differenza negativa di binary cross-entropy di validation indica una loss inferiore. Entrambi i pannelli riportano il valore del budget esteso meno quello del budget iniziale.

Le barre Random riguardano le teste sopra encoder casuali e ricevono la stessa estensione delle altre teste. Il confronto appare per ciascun probe appena termina il budget esteso. Le curve permettono di distinguere un miglioramento persistente della validation dall’eventuale crescita dell’errore dopo un minimo.

In [ ]:
if BASELINE_RESULTS is not None:
    baseline_results = load_results(BASELINE_RESULTS)
    plot_budget_gain(results, baseline_results)
    plt.show()
else:
    print('Questa serie contiene il budget iniziale di 30 epoche.')

## Apprendimento della testa

Le curve seguenti riguardano il primo modello completato della serie e il suo controllo casuale. L’asse orizzontale riporta le epoche di training della testa; l’asse verticale riporta la binary cross-entropy. La curva di training descrive i minibatch usati per ottimizzare la testa; quella di validation descrive l’applicazione della testa a una giornata separata. La selezione del checkpoint usa la mean IoU di validation, definita sopra.

Le curve conservano le prime 30 epoche e proseguono fino al budget corrente. La linea verticale indica il budget iniziale. Learning rate, seed, minibatch e selezione tramite validation restano gli stessi, per misurare l’effetto del numero di aggiornamenti della testa.

In [ ]:
selected_case = None
if not results.empty:
    available = results[results.kind == 'pretrained'].sort_values(['clips', 'case'])
    if len(available):
        selected_case = available.iloc[0]['case']
        info = json.loads((RESULTS / selected_case / 'comparison.json').read_text())
        fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
        for case, label in [(selected_case, 'Pretrained'), (info['paired_random_control'], 'Random')]:
            path = RESULTS / case / 'history.json'
            if path.exists():
                history = json.loads(path.read_text())
                epochs = [r['epoch'] for r in history]
                line, = ax.plot(epochs, [r['train_bce'] for r in history], label=f'{label}: training')
                ax.plot(epochs, [r['validation']['bce'] for r in history], '--', color=line.get_color(), label=f'{label}: validation')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Head training epoch'); ax.set_ylabel('Binary cross-entropy')
        if BASELINE_RESULTS is not None:
            ax.axvline(30, color='grey', linestyle=':', linewidth=1)
        ax.set_title('Cloud probe learning curves'); ax.legend(); ax.grid(alpha=.2)
        plt.show()
        print('Modello rappresentato:', selected_case)

## Esempi spaziali sul test

Gli esempi provengono da posizioni fissate nell’elenco ordinato delle clip di test. Le colonne mostrano IR 10.8 µm, il target CMA e le probabilità di nube del modello preaddestrato e del controllo casuale. Nel campo IR, la scala comune copre 210–300 K e i valori più freddi appaiono più chiari. Per target e probabilità, il bianco indica zero e il blu scuro uno. La maschera CMA descrive quindi due classi, mentre le predizioni mostrano il grado di probabilità assegnato alla nube. Date e posizioni dei ritagli sono riportate dopo la figura.

La testa decodifica separatamente i pixel di ciascuna patch. Questa struttura può produrre discontinuità lungo bordi distanziati di 16 pixel, visibili nelle mappe di probabilità. Il confronto mantiene la stessa testa per tutti gli encoder.

In [ ]:
if selected_case is not None:
    figure, example_metadata = plot_examples(RESULTS, selected_case)
    plt.show()
    for index, description in enumerate(example_metadata, 1):
        print(f'Esempio {index}: {description}')

## Portata del confronto

Questo primo esperimento valuta la decodificabilità della maschera CMA sulle rappresentazioni congelate. La copertura temporale del test, il seed unico e le approssimazioni del teacher definiscono l’estensione attuale dei risultati. Un’analisi successiva potrà aggiungere più giornate, seed, ablazioni temporali e spettrali e un encoder RGB adattato come controllo, per distinguere il contributo delle firme spettrali da quello della storia atmosferica.

I file `result.json` conservano anche precisione, richiamo, IoU del sereno, balanced accuracy e conteggi della matrice di confusione. `test_per_clip.json` conserva i risultati di ciascuna clip. Le metriche sono conteggi di pixel nella griglia proiettata del satellite.

Per verificare da SSH l’avanzamento e gli eventuali arresti:

```bash
python3 /home/isac/deep_convection/scripts/status_seviri_cloud_probes.py
```

La descrizione riproducibile del protocollo e i comandi di ripresa sono in `docs/SEVIRI_CLOUD_MASK_PROBE.md`.

## Three-head probing: cloud presence, cloud regime and convective evolution

The protocol in `three_heads_probing.md` adds three aligned segmentation targets to each sixteen-frame SEVIRI clip. **CMA** identifies clear and cloudy pixels. **CT**, the cloud-type target, groups the native classifications into clear sky, low cloud, mid-level cloud, high opaque cloud, and cirrus or multilayer cloud. **RDT-CW-inspired** describes background, cold non-convective cloud, triggering or growing convection, mature or decaying convection, and persistent overshooting tops. The latter are compact cloud-top features classified by the deterministic teacher as penetrating above its estimated anvil reference.

Each input contains the seven IR channels in their established order, sampled every five minutes at 224 × 224 pixels. Each head produces a classification for **all sixteen frames**, covering 75 minutes between the first and last observation. The preceding cloud-mask experiments used a smaller decoder and a target on one frame. Their results remain in the preceding section; comparisons within the new protocol share the same clips and targets.

The encoder attends across all sixteen input frames. Predictions for earlier frames can therefore use later observations within the clip; the final-frame output uses the available 75-minute history up to that time. The reported full-sequence scores measure retrospective segmentation of observed sequences.

Both teacher algorithms process the complete satellite domain causally before crops are extracted. Their states continue across consecutive observations, including midnight, and restart after an actual temporal gap. The first fifteen minutes provide temporal history. A common validity mask excludes missing observations and detected objects with insufficient tracking history. Background pixels use the observed fifteen-minute history directly, because the native object-tracking flags are assigned to detected cells.

The updated CT teacher and the RDT teacher use the same satellite observations and different classification procedures. RDT also uses its own internal cloud-type prefilter. Agreement between these algorithms measures the consistency of their pseudo-labels; physical validation against independent observations remains a separate evaluation.

### Exploratory dataset and teacher agreement

The protocol revision approved on **9 November 2026** uses **512 training clips from 1–2 July 2022, 192 validation clips from 4 July, and 192 test clips from 7 July**. Each clip contains sixteen observations at five-minute intervals, spanning 75 minutes. Every date touched by pretraining or its validation is excluded. At least one unused day separates the downstream periods; weather systems may persist across this interval. The comparisons therefore describe performance on these particular held-out periods.

Teacher acceptance requires at least **99.5%** of RDT convective pixels inside CMA clouds and fewer than **0.1%** in CT clear-sky or low-cloud classes. All July periods satisfy these requirements. CT high-opaque agreement is **87.1%, 86.9%, and 84.1%** for training, validation, and test. Under the approved exploratory protocol, this quantity is a reported diagnostic, and the dashed **90%** line denotes the original reference. The manifest preserves the original acceptance result and records the revised decision separately. All these fractions use full-domain observed pixels after warm-up, before exclusions for insufficient track history.

The corrected RDT teacher distinguishes measured radiometry from the cloud prefilter used to select candidate cells. Finite temperatures outside this prefilter contribute to spatial gradients. Actual missing observations retain their quality flags. A cell contributes targets when its trajectory has four observations at exact five-minute intervals and a finite temperature change relative to the actual fifteen-minute reference. Background pixels require four valid observations. This common validity mask applies to all three targets; native quality flags remain available for examining individual trend calculations.

The paired July verification covers **1,152 frames** with identical detected cold-cell footprints and CT maps. The fraction of cold-cell pixels retained as targets changes from 2.5% to 68.6% for training, 2.2% to 76.4% for validation, and 3.6% to 71.9% for test. The previous preparation used native temporal and phase flags; the corrected preparation checks the actual fifteen-minute trajectory. The coverage plot names the cold-cell population explicitly. The adjacent plot partitions all RDT convective pixels by CT class, showing where the two algorithms disagree. These comparisons measure agreement between deterministic pseudo-labels. Independent meteorological observations are needed to assess their physical accuracy.

Crop origins have individual-pixel resolution, with the complete 224 × 224 crop inside **20°W–40°E and 30°N–70°N**. Training draws assign **50%, 30%, and 20% probability** to groups A, B, and C. Group B contains at least 25% high-opaque or cirrus CT pixels and at least 1% cold non-convective RDT pixels among valid targets. A mixed scene qualifies when the central pixel in its last frame is a cold non-convective RDT cloud classified as high opaque or cirrus by CT. Convection elsewhere retains its class in the target maps. Convection-free scenes meeting the coverage requirements also qualify. Group B takes priority over A; remaining convective clips enter A, and remaining clips with less than 50% high-cloud coverage enter C. Within A, clips containing overshooting tops receive triple sampling weight. Validation and test use uniform spatial and temporal candidates.

The exhaustive sampling diagnostic examines **501,046 permitted origins in each of 35 July training windows**, checking all input channels and frames. It finds **21,911 complete cold-centred candidates** for the revised group B. The plot of the original convection-free group reports zero July candidates, explaining the sampling revision. Counts refer to correlated position/window combinations, including overlapping crops. A single mean and standard deviation fitted on downstream training pixels and channels normalize every encoder control. The manifest records those statistics, class support and rejected inputs.

Additional seasonal diagnostics are retained below as context. The nine corrected-teacher periods all fall below the original CT reference; February also exceeds the retained clear/low threshold. Each phase-conditioned percentage uses pixels of the named native RDT phase as its denominator. In July, mature cells reach about 94% CT high-opaque agreement and growing cells 72–76%. Overshooting-top support is small and includes repeated observations of tracked features; grey cells indicate absent phases. The spatial revision example uses 7 July at 12:00 UTC: blue in its class-change panel marks a revised RDT label, while blue in the validity panels marks retained targets.

The published training set contains **256 group-A, 154 group-B, and 102 group-C clips**; every selected B clip is a mixed scene with convection elsewhere. The shared normalization has mean **263.6 K** and standard deviation **21.1 K**. The class-support plot reports the fraction of valid target pixels in each split, using a logarithmic vertical axis for RDT to display its rare classes. Overshooting-top support comprises **441 training, 83 validation, and 91 test pixel observations**, including repeated observations and overlap between crops. This support limits the stability of class-specific OT comparisons.

In [ ]:
from pathlib import Path
import importlib, json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from emma_gpm import three_head_study
three = importlib.reload(three_head_study)
THREE_ROOT = Path("/data1/deep_convection_results/three_head_probes_20261109")
THREE_FIGURES = THREE_ROOT / "notebook_figures"
THREE_FIGURES.mkdir(parents=True, exist_ok=True)
THREE_SUITE = three.load_suite(THREE_ROOT)
def show_three(fig, filename):
    if fig is not None:
        fig.savefig(THREE_FIGURES / filename, dpi=130, bbox_inches="tight")
        display(fig)
        plt.close(fig)
display(Markdown(three.status_text(THREE_SUITE)))
preparation_status = THREE_ROOT / "preparation_status.json"
if preparation_status.exists():
    preparation = json.loads(preparation_status.read_text())
    if not preparation.get("training_started", False):
        display(Markdown("**Preparation status:** " + preparation["message"]))
show_three(three.teacher_gate_figure({
    "July 2022 · v2": "/data1/deep_convection_data/three_head_teachers_summer2022_20261109",
    "July 2022 · corrected v3": "/data1/deep_convection_data/three_head_teachers_summer2022_history15m_v2_20261109",
    "Other seasons · v3": "/data1/deep_convection_data/three_head_teachers_seasonal2022_history15m_v2_20261109",
    "Autumn 2021 · v3": "/data1/deep_convection_data/three_head_teachers_autumn2021_history15m_v2_20261109",
}), "teacher_acceptance.png")
show_three(three.dataset_support_figure(THREE_SUITE), "three_head_target_support.png")
three_scores = three.metric_frame(THREE_SUITE)
if not three_scores.empty:
    three_scores.to_csv(THREE_ROOT / "test_class_metrics.csv", index=False)

revision_report = Path("/data1/deep_convection_results/three_head_probe_integration_20261109/teacher_revision_verification.json")
if revision_report.exists():
    revision = json.loads(revision_report.read_text())
    show_three(three.teacher_revision_figure(revision), "teacher_revision_complete_periods.png")

revision_maps = Path("/data1/deep_convection_results/three_head_probe_integration_20261109/teacher_revision_maps.png")
if revision_maps.exists():
    from IPython.display import Image
    display(Image(filename=str(revision_maps)))

show_three(three.sampling_feasibility_figure({
    "10–11 February": "/data1/deep_convection_results/three_head_probe_integration_20261109/sampling_v3_february_radiometric.json",
    "4 April": "/data1/deep_convection_results/three_head_probe_integration_20261109/sampling_v3_april_radiometric.json",
    "1–2 July": "/data1/deep_convection_results/three_head_probe_integration_20261109/sampling_v3_july_radiometric.json",
}), "strict_sampling_feasibility.png")

show_three(three.native_phase_agreement_figure({
    "28 Oct 2021": "/data1/deep_convection_data/three_head_teachers_autumn2021_history15m_v2_20261109/train",
    "30 Oct 2021": "/data1/deep_convection_data/three_head_teachers_autumn2021_history15m_v2_20261109/validation",
    "8 Nov 2021": "/data1/deep_convection_data/three_head_teachers_autumn2021_history15m_v2_20261109/test",
    "10–11 Feb 2022": "/data1/deep_convection_data/three_head_teachers_seasonal2022_history15m_v2_20261109/train",
    "4 Apr 2022": "/data1/deep_convection_data/three_head_teachers_seasonal2022_history15m_v2_20261109/validation",
    "1 May 2022": "/data1/deep_convection_data/three_head_teachers_seasonal2022_history15m_v2_20261109/test",
    "1–2 Jul 2022": "/data1/deep_convection_data/three_head_teachers_summer2022_history15m_v2_20261109/train",
    "4 Jul 2022": "/data1/deep_convection_data/three_head_teachers_summer2022_history15m_v2_20261109/validation",
    "7 Jul 2022": "/data1/deep_convection_data/three_head_teachers_summer2022_history15m_v2_20261109/test",
}), "native_phase_ct_agreement.png")


### Frozen encoder and shared decoder

The main comparison starts from the Small MVM encoder pretrained on 65,536 SEVIRI clips. The checkpoint with the lowest pretraining validation loss supplies the weights. Its learned parameters are loaded strictly; the fixed sinusoidal position sequence is rebuilt for sixteen frames because the historical SSL experiments encoded ten context frames. This adaptation preserves the learned convolution and Transformer weights.

Features from Transformer blocks 3, 6, 9 and 12 feed a shared 3D Feature Pyramid Network, or FPN, which combines representations from several depths and progressively restores spatial resolution. Its shared feature width is 128. The CMA head predicts cloud probability. CT receives that probability alongside the shared features; RDT receives both the CMA probability and the five CT probabilities. Independent single-task probes use a separate FPN for each target.

CMA training combines binary cross-entropy and Dice loss. CT uses focal cross-entropy and multiclass Dice. RDT uses class-weighted focal cross-entropy and generalized Dice, with weights fitted on training support. Joint training adds a soft consistency penalty when predicted convection probability exceeds high-opaque-cloud probability or predicted CT cloud probability exceeds CMA cloud probability. The penalty coefficient is 0.1. Each target contributes with unit weight, and invalid pixels are excluded from every loss term.

The FPN implementation projects its shared features to head logits before the final linear upsampling. Pointwise linear projection and linear interpolation commute, allowing the same full-resolution prediction function with much smaller intermediate tensors. Hierarchical conditioning is applied to full-resolution probabilities. A numerical test compares this computation with explicit full-resolution feature construction.

**E0-Rand** uses a frozen random Small encoder. **E0-RGB** uses a frozen Small encoder transferred from natural videos: the available checkpoint includes MAE pretraining and supervised Kinetics fine-tuning. Its RGB projection is averaged and adapted to seven input channels. **E1-Single-cma**, **E1-Single-ct** and **E1-Single-rdt** train independent decoders. **E1-Joint3** trains the three conditioned heads together. These labels identify the corresponding curves and markers below. Every run initializes fresh decoder weights with the same seed, and frozen-encoder weights are checked before and after training.

The initial budget is 60 epochs per experiment, with best-checkpoint selection by validation loss. Training checkpoints preserve model, optimizer, scheduler and random-number states. Test results are computed after selection. Each loss panel below compares the same target-specific loss; joint total loss sums several target terms and has a different scale.

In [ ]:
FROZEN_NAMES = ["E0-Rand", "E0-RGB", "E1-Single-cma", "E1-Single-ct", "E1-Single-rdt", "E1-Joint3"]
show_three(three.learning_figure(THREE_SUITE), "frozen_validation_losses.png")
show_three(three.comparison_figure(THREE_SUITE, FROZEN_NAMES, "Frozen-encoder class agreement"), "frozen_class_iou.png")
show_three(three.comparison_figure(THREE_SUITE, FROZEN_NAMES, "Frozen-encoder class F1", metric="f1"), "frozen_class_f1.png")
show_three(three.active_convection_figure(THREE_SUITE, FROZEN_NAMES), "frozen_active_convection_csi.png")
show_three(three.brier_figure(THREE_SUITE, FROZEN_NAMES), "frozen_rdt_brier.png")
current_result_summary = three.result_summary_text(THREE_SUITE)
if current_result_summary:
    display(Markdown(current_result_summary))


The class intersection-over-union, **IoU**, measures the overlap between predicted and teacher-labelled pixels relative to their combined area. For each class this is also the **critical success index, CSI**. **F1** combines class precision and recall. Their ideal value is one. The Brier score measures the mean squared error of predicted probabilities; its ideal value is zero. The multiclass Brier score sums the five class contributions for each pixel.

Active convection combines RDT classes triggering/growing, mature/decaying and persistent overshooting top. Its binary prediction uses the sum of their probabilities and a threshold of 0.5. Class-wise scores use the most probable individual class. Metrics pool pixel counts over the entire held-out test split. The exported results include support and confusion counts; a class with no observed or predicted support has an undefined overlap score. In particular, the overshooting-top result must be interpreted together with its actual target support.

### Sequential adaptation and transfer controls

The sequential comparison begins with the independent frozen RDT probe, labelled phase 1. Phase 2 adapts the encoder to CMA, freezes it again, and trains new independent CT and RDT decoders. Phase 3 starts from the CMA-adapted encoder, adapts it to CT, freezes it, and trains another new RDT decoder. Every phase uses the same downstream training, validation and test split.

The **selective** variant updates the last four Transformer blocks with encoder learning rate 0.00001 and decoder learning rate 0.001, while penalizing displacement from the original SSL weights. The **cumulative** variant uses the same selective unfreezing: during CMA adaptation it retains an auxiliary masked-reconstruction loss with coefficient 0.1, initialized from the original MVM decoder; during CT adaptation it retains the learned CMA head and shared FPN. The fresh frozen probes following either adaptation stage are initialized independently. **E4-JointFT** updates the complete encoder and all three heads together and provides a separate reference for end-to-end adaptation.

The sequential plot follows active-convection CSI across the three phases. An increase indicates that the intermediate task made the fixed RDT target more accessible to a newly trained decoder. A decrease can motivate the temporal interventions below; distinguishing temporal forgetting from other changes also requires inspecting those intervention results.

In [ ]:
show_three(three.sequential_figure(THREE_SUITE), "sequential_active_convection.png")
SEQUENTIAL_NAMES = ["E1-Single-rdt", "E1-Joint3", "E2-selective-rdt", "E3-selective-rdt", "E2-cumulative-rdt", "E3-cumulative-rdt", "E4-JointFT"]
show_three(three.comparison_figure(THREE_SUITE, SEQUENTIAL_NAMES, "Sequential adaptation and joint fine-tuning"), "sequential_class_iou.png")
show_three(three.brier_figure(THREE_SUITE, SEQUENTIAL_NAMES), "sequential_rdt_brier.png")

### Temporal, spectral and pretraining comparisons

Diagnostic evaluation keeps the selected model and test targets fixed while changing the input. Temporal permutation changes frame order; repeating the last frame removes observed evolution. Spectral interventions retain WV 6.2 plus IR 10.8, retain both water-vapour channels plus IR 10.8, or suppress only IR 13.4. Suppressed channels are set to the global training mean. These interventions are applied only during the diagnostic evaluation. Their CSI changes quantify sensitivity to the altered input and include the effect of input distribution shift. Evaluation pools all sixteen output frames against targets in their original chronological slots. Temporal permutation therefore combines sensitivity to sequence structure with disruption of frame–target correspondence; repeating the last frame also changes the observed cloud geometry associated with earlier targets.

The pretraining comparison attaches the same three-head decoder to Small encoders trained with MVM, Future, Joint and Residual on 65,536 clips, and to the Base Future encoder trained on 131,072 clips on Leonardo. These encoders share the downstream dataset and decoder-training protocol. The Base–Small comparison combines changes in architecture, pretraining sample selection and pretraining optimization budget; it measures the resulting representations under this downstream evaluation.

In [ ]:
show_three(three.ablation_figure(THREE_SUITE), "temporal_spectral_interventions.png")
TRANSFER_NAMES = ["E1-Joint3", "Compare-Small-65K-future", "Compare-Small-65K-joint", "Compare-Small-65K-residual", "Compare-Base-131K-future"]
show_three(three.comparison_figure(THREE_SUITE, TRANSFER_NAMES, "Pretraining representations with a common downstream decoder"), "pretraining_transfer_iou.png")
show_three(three.active_convection_figure(THREE_SUITE, TRANSFER_NAMES, "Active convection across pretrained encoders"), "pretraining_active_convection_csi.png")
show_three(three.brier_figure(THREE_SUITE, TRANSFER_NAMES), "pretraining_transfer_brier.png")

### Aligned segmentation examples

The paired images show teacher targets and predictions from the selected validation checkpoint on the same held-out clip and its final frame. The colour legends define each class directly. Grey pixels are excluded by the common validity mask. The first scene has the largest teacher-labelled convective area in the final frame among the eight preselected test examples. The second is the first saved example whose final frame contains RDT background only, including clear sky and warmer clouds. Selection uses teacher labels alone. Encoder comparisons use the same clips and frame. The reported metrics pool the complete test split.

In [ ]:
show_three(three.example_figure(THREE_SUITE, "E1-Joint3"), "joint3_convective_example.png")
show_three(three.example_figure(THREE_SUITE, "E1-Joint3", scene="background"), "joint3_background_example.png")
show_three(three.example_figure(THREE_SUITE, "Compare-Base-131K-future"), "leonardo_encoder_example.png")